In [1]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

## Clear GPU cache

In [15]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [16]:
df = pd.read_csv('data/news_summary_more.csv', encoding='latin-1')
df = df[['headlines', 'text']].dropna().reset_index(drop=True)
df.columns = ['headline', 'article']

print(f"Total rows: {len(df)}")
df.head(3)

Total rows: 98401


,headline,article
0,upGrad learner switches to career in ML & Al w...,"Saurav Kant, an alumnus of upGrad and IIIT-B's..."
1,Delhi techie wins free food from Swiggy for on...,Kunal Shah's credit card bill payment platform...
2,New Zealand end Rohit Sharma-led India's 12-ma...,New Zealand defeated India by 8 wickets in the...


In [17]:
article_len = df['article'].str.split().apply(len)
headline_len = df['headline'].str.split().apply(len)

print(f"Article  — mean: {article_len.mean():.1f}, 95th pct: {article_len.quantile(0.95):.0f}")
print(f"Headline — mean: {headline_len.mean():.1f}, 95th pct: {headline_len.quantile(0.95):.0f}")

Article  — mean: 58.2, 95th pct: 60
Headline — mean: 9.6, 95th pct: 12


## Tokenization

In [18]:
import nltk
from nltk.tokenize import word_tokenize

def tokenize(text):
    return word_tokenize(str(text).lower())

MAX_ARTICLE_LEN = 70
MAX_HEADLINE_LEN = 15
MIN_LEN = 3

df['article_tokens'] = df['article'].apply(tokenize)
df['headline_tokens'] = df['headline'].apply(tokenize)

mask = (
    df['article_tokens'].apply(len).between(MIN_LEN, MAX_ARTICLE_LEN) &
    df['headline_tokens'].apply(len).between(MIN_LEN, MAX_HEADLINE_LEN)
)
df = df[mask].reset_index(drop=True)

print(f"After filtering: {len(df)} pairs")

After filtering: 77122 pairs


## Build vocab

In [19]:
from collections import Counter

MIN_FREQ = 2

def build_vocab(token_lists, specials):
    counter = Counter()
    for toks in token_lists:
        counter.update(toks)
    itos = list(specials) + [t for t, c in counter.most_common() if c >= MIN_FREQ]
    stoi = {t: i for i, t in enumerate(itos)}
    return stoi, itos

src_stoi, src_itos = build_vocab(df['article_tokens'], ['<pad>', '<unk>'])
tgt_stoi, tgt_itos = build_vocab(df['headline_tokens'], ['<pad>', '<unk>', '<sos>', '<eos>'])

SRC_PAD, SRC_UNK = src_stoi['<pad>'], src_stoi['<unk>']
TGT_PAD = tgt_stoi['<pad>']
TGT_UNK = tgt_stoi['<unk>']
TGT_SOS = tgt_stoi['<sos>']
TGT_EOS = tgt_stoi['<eos>']

SRC_VOCAB = len(src_itos)
TGT_VOCAB = len(tgt_itos)

print(f"Source vocab: {SRC_VOCAB}")
print(f"Target vocab: {TGT_VOCAB}")

Source vocab: 54154
Target vocab: 21991


In [20]:
from torch.utils.data import Dataset, DataLoader

from torch.nn.utils.rnn import pad_sequence

class SummaryDataset(Dataset):
    def __init__(self, src_tokens, tgt_tokens):
        self.src = src_tokens.tolist()
        self.tgt = tgt_tokens.tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        s = [src_stoi.get(t, SRC_UNK) for t in self.src[idx]]
        t = [TGT_SOS] + [tgt_stoi.get(w, TGT_UNK) for w in self.tgt[idx]] + [TGT_EOS]
        return torch.tensor(s, dtype=torch.long), torch.tensor(t, dtype=torch.long)


def collate_batch(batch):
    srcs, tgts = zip(*batch)
    src_padded = pad_sequence(srcs, batch_first=True, padding_value=SRC_PAD)
    tgt_padded = pad_sequence(tgts, batch_first=True, padding_value=TGT_PAD)
    tgt_in = tgt_padded[:, :-1]
    tgt_out = tgt_padded[:, 1:]
    return src_padded, tgt_in, tgt_out




In [21]:
shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

test_size = int(len(shuffled) * 0.05)
val_size = int(len(shuffled) * 0.05)

test_df = shuffled.iloc[:test_size].reset_index(drop=True)
val_df = shuffled.iloc[test_size:test_size + val_size].reset_index(drop=True)
tr_df = shuffled.iloc[test_size + val_size:].reset_index(drop=True)

train_ds = SummaryDataset(tr_df["article_tokens"], tr_df["headline_tokens"])
val_ds = SummaryDataset(val_df["article_tokens"], val_df["headline_tokens"])
test_ds = SummaryDataset(test_df["article_tokens"], test_df["headline_tokens"])

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

print(f"Train pairs: {len(train_ds)}, Val pairs: {len(val_ds)}, Test pairs: {len(test_ds)}")

Train pairs: 69410, Val pairs: 3856, Test pairs: 3856


## Model

In [22]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)

    def forward(self, src):
        emb = self.embedding(src)
        _, (h, c) = self.lstm(emb)
        return h, c


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt_in, hidden, cell):
        emb = self.embedding(tgt_in)
        out, (h, c) = self.lstm(emb, (hidden, cell))
        return self.fc(out), h, c


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt_in):
        h, c = self.encoder(src)
        logits, _, _ = self.decoder(tgt_in, h, c)
        return logits


EMB = 256
HID = 512
LAYERS = 2
DROPOUT = 0.4 

encoder = Encoder(SRC_VOCAB, EMB, HID, LAYERS, SRC_PAD, DROPOUT)
decoder = Decoder(TGT_VOCAB, EMB, HID, LAYERS, TGT_PAD, DROPOUT)
model = Seq2Seq(encoder, decoder).to(device)

print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(54154, 256, padding_idx=0)
    (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.4)
  )
  (decoder): Decoder(
    (embedding): Embedding(21991, 256, padding_idx=0)
    (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.4)
    (fc): Linear(in_features=512, out_features=21991, bias=True)
  )
)


## Training

In [23]:

EPOCHS = 11
LR = 0.001

criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_tokens = 0.0, 0
    for src, tgt_in, tgt_out in train_loader:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
        logits = model(src, tgt_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        n = (tgt_out != TGT_PAD).sum().item()
        train_loss += loss.item() * n
        train_tokens += n

    train_loss /= train_tokens

    model.eval()
    val_loss, val_tokens = 0.0, 0
    with torch.no_grad():
        for src, tgt_in, tgt_out in val_loader:
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            n = (tgt_out != TGT_PAD).sum().item()
            val_loss += loss.item() * n
            val_tokens += n

    val_loss /= val_tokens

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")



Epoch 1: train_loss=7.0171, val_loss=6.6312
Epoch 2: train_loss=6.3800, val_loss=6.2625
Epoch 3: train_loss=5.9742, val_loss=5.9205
Epoch 4: train_loss=5.5522, val_loss=5.5803
Epoch 5: train_loss=5.1369, val_loss=5.3287
Epoch 6: train_loss=4.7938, val_loss=5.1482
Epoch 7: train_loss=4.5064, val_loss=5.0207
Epoch 8: train_loss=4.2506, val_loss=4.9498
Epoch 9: train_loss=4.0168, val_loss=4.8758
Epoch 10: train_loss=3.8010, val_loss=4.8518
Epoch 11: train_loss=3.6005, val_loss=4.8280


In [24]:
@torch.no_grad()
def generate(article_text, max_len=MAX_HEADLINE_LEN + 2):
    model.eval()
    toks = tokenize(article_text)[:MAX_ARTICLE_LEN]
    ids = [src_stoi.get(t, SRC_UNK) for t in toks]
    if not ids:
        return ''

    src = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    h, c = model.encoder(src)

    input_tok = torch.tensor([[TGT_SOS]], dtype=torch.long, device=device)
    generated = []
    for _ in range(max_len):
        logits, h, c = model.decoder(input_tok, h, c)
        next_id = logits[0, -1].argmax().item()
        if next_id == TGT_EOS:
            break
        generated.append(next_id)
        input_tok = torch.tensor([[next_id]], dtype=torch.long, device=device)

    return ' '.join(tgt_itos[i] for i in generated)

## Testing

In [25]:
model.eval()
test_loss, test_tokens = 0.0, 0
with torch.no_grad():
    for src, tgt_in, tgt_out in test_loader:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
        logits = model(src, tgt_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        n = (tgt_out != TGT_PAD).sum().item()
        test_loss += loss.item() * n
        test_tokens += n

test_loss /= test_tokens
print(f"Test loss: {test_loss:.4f}")

Test loss: 4.7834


In [26]:
sample_idx = np.random.RandomState(SEED).choice(len(test_df), size=6, replace=False)

for i in sample_idx:
    article = test_df['article'].iloc[i]
    true_headline = test_df['headline'].iloc[i]
    pred_headline = generate(article)

    print(f"ARTICLE : {article[:200]}{'...' if len(article) > 200 else ''}")
    print(f"TRUE    : {true_headline}")
    print(f"PRED    : {pred_headline}")
    print('-' * 100)

ARTICLE : The Jharkhand High Court has extended provisional bail awarded to RJD supremo Lalu Prasad Yadav by six weeks till August 17 on medical grounds. Stating that doctors recommended two-month rest to Lalu ...
TRUE    : Court extends bail to Lalu by 6 weeks on medical grounds
PRED    : lalu 's son tej pratap yadav granted bail in kerala
----------------------------------------------------------------------------------------------------
ARTICLE : Former cable news anchor Greta Van Susteren has developed an app called 'Sorry' to allow users to send apologies. The app will let users accept or reject an apology sent by other users which will be k...
TRUE    : Woman develops 'Sorry' app for users to send apologies
PRED    : youtube app lets users share ' hey ' feature to videos
----------------------------------------------------------------------------------------------------
ARTICLE : Two cinema halls were gutted after a massive fire broke out at a cinema complex in Visakhapatnam's Ga

## Custom articles

In [27]:
custom_articles = [
    "The Reserve Bank of India kept its key policy interest rate unchanged on Friday, holding the repo rate at 6.5% for the fifth straight meeting as inflation remains within target.",
    "Scientists at NASA announced the discovery of a new Earth-sized exoplanet orbiting a nearby star in what appears to be the habitable zone where liquid water could exist.",
    "India defeated Australia by six wickets in the third one-day international at Rajkot on Wednesday to take an unassailable lead in the five-match series.",
]

for a in custom_articles:
    print(f"ARTICLE : {a}")
    print(f"PRED    : {generate(a)}")
    print('-' * 100)

ARTICLE : The Reserve Bank of India kept its key policy interest rate unchanged on Friday, holding the repo rate at 6.5% for the fifth straight meeting as inflation remains within target.
PRED    : rbi keeps repo rate unchanged for the first time
----------------------------------------------------------------------------------------------------
ARTICLE : Scientists at NASA announced the discovery of a new Earth-sized exoplanet orbiting a nearby star in what appears to be the habitable zone where liquid water could exist.
PRED    : nasa 's first ever <unk> <unk> <unk> <unk>
----------------------------------------------------------------------------------------------------
ARTICLE : India defeated Australia by six wickets in the third one-day international at Rajkot on Wednesday to take an unassailable lead in the five-match series.
PRED    : india defeat australia to win their first time
--------------------------------------------------------------------------------------------------